In [ ]:
import pandas as pd

cols = ['cian_id','source_table','first_seen_at','last_seen_at','consecutive_misses','snapshot_date',
        'hist_price','hist_recorded_at','price','price_per_m2','deal_conditions','region','municipality',
        'district','rooms','total_area','living_area','kitchen_area','floor','total_floors','ceiling_height',
        'renovation','bathrooms','balcony','window_view','is_apartments','year_built','building_type','parking',
        'is_new_building','developer','residential_complex','completion_date','seller_type','phone_protected',
        'publication_date']

# загружаем датасет, приводим типы к дате
df = pd.read_csv('./data/listings.csv', usecols=cols)
for c in ['first_seen_at','last_seen_at','snapshot_date','hist_recorded_at']:
    df[c] = pd.to_datetime(df[c], utc=True)

# маркируем "живые" объявления, выставляем маркеры свежести (при обработке отсортировать по misses > 0)
df['live'] = df['source_table'] == 'live'
g = df.groupby('cian_id')
life = pd.DataFrame({
    'first_seen': g['first_seen_at'].min(),
    'last_seen': g['last_seen_at'].max(),
    'sold': (~g['live'].max()).astype(int),
    'misses': g['consecutive_misses'].max(), 
})

# таргет 1 - "дней на рынке"
life['days_on_market'] = (life['last_seen'] - life['first_seen']).dt.days

In [ ]:
static_cols = ['price','price_per_m2','deal_conditions','region','municipality','district','rooms',
               'total_area','living_area','kitchen_area','floor','total_floors','ceiling_height','renovation',
               'bathrooms','balcony','window_view','is_apartments','year_built','building_type','parking',
               'is_new_building','developer','residential_complex','completion_date','seller_type',
               'phone_protected','publication_date']

# оставляем только последнее состояние квартиры
static = df.sort_values('snapshot_date').groupby('cian_id')[static_cols].last()

# убиваем дубликаты и формируем первую и последнюю цену
ph = df[df['hist_recorded_at'].notna()].drop_duplicates(['cian_id','hist_recorded_at','hist_price'])
ph = ph.sort_values('hist_recorded_at').groupby('cian_id')['hist_price']
price = pd.DataFrame({'price_first': ph.first(), 'price_last': ph.last()})

# прицепляем цены обратно
out = life.join(static).join(price).reset_index()
# убираем совсем невалидные записи
out = out[(out['days_on_market'] >= 0) & (out['total_area'] > 0) & (out['price'] > 0)]

QUANTILE = 0.99
pct_cols = ['price', 'price_per_m2', 'total_area', 'living_area', 'kitchen_area', 'price_first', 'price_last']
for c in pct_cols:
    lo = out[c].quantile(1 - QUANTILE)
    hi = out[c].quantile(QUANTILE)
    out = out[out[c].isna() | out[c].between(lo, hi)]

out.to_csv('./data/listings_preprocessed.csv', index=False)
out
